# Kaggle Submission: exp_011 HGNetV2 4-Fold Ensemble

Raw 4-fold ensemble for the repository-native `exp_011` branch. This notebook restores the four soundscape-aware checkpoints, runs 5-second inference on the hidden test soundscapes, and writes `/kaggle/working/submission.csv`.


## Attach Before Running

- BirdCLEF+ 2026 competition data
- One Kaggle model dataset containing:
  - `fold_00/best_model.pt`
  - `fold_01/best_model.pt`
  - `fold_02/best_model.pt`
  - `fold_03/best_model.pt`

Use `Internet = Off`. GPU is recommended.


In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import defaultdict
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
import typing as tp

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from tqdm.auto import tqdm


In [ ]:
MODEL_DATASET_HINT = None  # Example: 'birdclef-exp011-hgnetv2-4fold'
CHECKPOINT_NAME = 'best_model.pt'
FOLD_IDS = (0, 1, 2, 3)
AUDIO_SUFFIXES = {'.ogg', '.wav', '.flac', '.mp3'}


@dataclass
class Config:
    sample_rate: int = 32_000
    segment_seconds: int = 5
    n_fft: int = 2048
    win_length: int = 626
    hop_length: int = 313
    f_min: int = 20
    n_mels: int = 256
    top_db: float = 80.0
    image_size: tuple[int, int] = (256, 256)
    model_name: str = 'hgnetv2_b0.ssld_stage2_ft_in1k'
    batch_chunks: int = 48

    @property
    def chunk_samples(self) -> int:
        return int(self.sample_rate * self.segment_seconds)


CFG = Config()
INPUT_ROOT = Path('/kaggle/input')
ROW_PATTERN = re.compile(r'^(.*)_(\d+)$')


def resolve_project_root() -> Path | None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'birdclef-2026').exists():
            return candidate
    return None


PROJECT_ROOT = resolve_project_root()
if Path('/kaggle/working').exists():
    WORK_ROOT = Path('/kaggle/working')
else:
    WORK_ROOT = (PROJECT_ROOT / 'submissions' / 'debug' / 'exp_011_hgnetv2_4fold') if PROJECT_ROOT else Path.cwd()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORK_ROOT / 'submission.csv'


In [ ]:
def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


device = pick_device()
amp_enabled = device.type == 'cuda'
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')


def autocast_context():
    if amp_enabled:
        return torch.amp.autocast('cuda', enabled=True)
    return nullcontext()


def resolve_data_root() -> Path:
    direct_candidates = [
        INPUT_ROOT / 'competitions' / 'birdclef-2026',
        INPUT_ROOT / 'birdclef-2026',
    ]
    for candidate in direct_candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate

    if INPUT_ROOT.exists():
        for child in sorted(INPUT_ROOT.iterdir()):
            if (child / 'sample_submission.csv').exists():
                return child
            matches = sorted(child.rglob('sample_submission.csv'))
            if matches:
                return matches[0].parent

    if PROJECT_ROOT is not None:
        local_data = PROJECT_ROOT / 'data' / 'birdclef-2026'
        if (local_data / 'sample_submission.csv').exists():
            return local_data

    raise FileNotFoundError('BirdCLEF competition data was not found.')


def iter_candidate_roots(model_dataset_hint: str | None):
    if model_dataset_hint:
        hinted_path = Path(model_dataset_hint)
        if hinted_path.exists():
            yield hinted_path
        else:
            hinted_under_input = INPUT_ROOT / model_dataset_hint
            if hinted_under_input.exists():
                yield hinted_under_input

    if PROJECT_ROOT is not None:
        local_packaged = PROJECT_ROOT / 'submissions' / 'kaggle_datasets' / 'birdclef-exp011-hgnetv2-3fold'
        if local_packaged.exists():
            yield local_packaged
        local_outputs = PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_011_hgnetv2_soundscape_supervised'
        if local_outputs.exists():
            yield local_outputs

    if INPUT_ROOT.exists():
        for child in sorted(INPUT_ROOT.iterdir()):
            yield child


def resolve_fold_checkpoint_paths(model_dataset_hint: str | None = None, checkpoint_name: str = CHECKPOINT_NAME, fold_ids=FOLD_IDS) -> dict[int, Path]:
    searched = []
    seen = set()
    found: dict[int, Path] = {}

    for root in iter_candidate_roots(model_dataset_hint):
        try:
            root = root.resolve()
        except FileNotFoundError:
            continue

        key = str(root)
        if key in seen:
            continue
        seen.add(key)
        searched.append(key)

        if root.is_dir():
            for path in sorted(root.rglob(checkpoint_name)):
                fold = None
                for part in path.parts:
                    if re.fullmatch(r'fold_\d{2}', part):
                        fold = int(part.split('_')[1])
                        break
                if fold in fold_ids and fold not in found:
                    found[int(fold)] = path
        elif root.is_file() and root.name == checkpoint_name:
            fold = None
            for part in root.parts:
                if re.fullmatch(r'fold_\d{2}', part):
                    fold = int(part.split('_')[1])
                    break
            if fold in fold_ids and fold not in found:
                found[int(fold)] = root

        if len(found) == len(tuple(fold_ids)):
            break

    missing = [int(fold) for fold in fold_ids if int(fold) not in found]
    if missing:
        raise FileNotFoundError(f'Could not find fold checkpoints for folds {missing}. Searched roots: {searched[:12]}')
    return {int(fold): found[int(fold)] for fold in fold_ids}


In [ ]:
DATA_ROOT = resolve_data_root()
sample_sub = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
taxonomy = pd.read_csv(DATA_ROOT / 'taxonomy.csv')
CLASSES = taxonomy['primary_label'].tolist()
class_to_idx = {label: idx for idx, label in enumerate(CLASSES)}
submission_species = sample_sub.columns[1:].tolist()
missing_in_taxonomy = [label for label in submission_species if label not in class_to_idx]
if missing_in_taxonomy:
    raise ValueError(f'Submission labels missing in taxonomy: {missing_in_taxonomy[:8]}')
submission_class_indices = np.asarray([class_to_idx[label] for label in submission_species], dtype=np.int64)


def parse_row_id(row_id: str) -> tuple[str | None, int | None]:
    match = ROW_PATTERN.match(str(row_id))
    if not match:
        return None, None
    return match.group(1), int(match.group(2))


def build_expected_stem_map(row_ids: tp.Iterable[str]) -> dict[str, list[int]]:
    grouped: dict[str, list[int]] = defaultdict(list)
    for row_id in row_ids:
        stem, end_second = parse_row_id(row_id)
        if stem is None or end_second is None:
            continue
        grouped[stem].append(int(end_second))
    return {stem: sorted(set(seconds)) for stem, seconds in grouped.items()}


expected_seconds_by_stem = build_expected_stem_map(sample_sub['row_id'].tolist())
expected_stems = set(expected_seconds_by_stem)
print({
    'data_root': str(DATA_ROOT),
    'device': str(device),
    'n_classes': len(CLASSES),
    'submission_species': len(submission_species),
    'expected_soundscapes': len(expected_stems),
})


In [ ]:
def discover_test_files(data_root: Path, expected_stems: set[str]) -> tuple[dict[str, Path], bool, pd.DataFrame]:
    candidate_roots = [
        data_root / 'test_soundscapes',
        data_root / 'test_audio',
        data_root,
        INPUT_ROOT,
    ]
    discovered: dict[str, Path] = {}
    seen = set()
    for root in candidate_roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob('*')):
            if not path.is_file() or path.suffix.lower() not in AUDIO_SUFFIXES:
                continue
            stem = path.stem
            if stem in expected_stems and stem not in discovered:
                discovered[stem] = path
        if len(discovered) == len(expected_stems):
            break

    if discovered:
        return discovered, False, sample_sub.copy()

    train_sc_dir = data_root / 'train_soundscapes'
    if not train_sc_dir.exists():
        raise FileNotFoundError('No hidden test files found, and train_soundscapes fallback is unavailable.')

    fallback_paths = sorted([p for p in train_sc_dir.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES])[:4]
    if not fallback_paths:
        raise FileNotFoundError('No soundscape files found for inference.')

    fallback_rows = []
    fallback_map = {}
    for path in fallback_paths:
        info = sf.info(str(path))
        duration_sec = int(math.ceil(info.frames / max(1, info.samplerate)))
        end_seconds = list(range(CFG.segment_seconds, duration_sec + 1, CFG.segment_seconds))
        if not end_seconds:
            end_seconds = [CFG.segment_seconds]
        fallback_map[path.stem] = path
        for end_second in end_seconds:
            fallback_rows.append({'row_id': f'{path.stem}_{end_second}'})

    fallback_sub = pd.DataFrame({'row_id': [row['row_id'] for row in fallback_rows]})
    zeros = pd.DataFrame(
        np.zeros((len(fallback_sub), len(submission_species)), dtype=np.float32),
        columns=submission_species,
    )
    fallback_sub = pd.concat([fallback_sub, zeros], axis=1)
    print('Warning: hidden test files not found; using a small train fallback for a dry run.')
    return fallback_map, True, fallback_sub


test_file_map, used_test_fallback, active_sample_sub = discover_test_files(DATA_ROOT, expected_stems)
active_expected_seconds = build_expected_stem_map(active_sample_sub['row_id'].tolist())
print({
    'used_test_fallback': used_test_fallback,
    'matched_test_stems': len(test_file_map),
    'expected_row_ids': int(len(active_sample_sub)),
})
first_path = next(iter(test_file_map.values()))
print('First test path:', first_path)


In [ ]:
class LogMelSpectrogramTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate,
            n_fft=CFG.n_fft,
            win_length=CFG.win_length,
            hop_length=CFG.hop_length,
            f_min=CFG.f_min,
            n_mels=CFG.n_mels,
            power=2.0,
            center=True,
            pad_mode='reflect',
            norm='slaney',
            mel_scale='htk',
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=CFG.top_db)

    @torch.no_grad()
    def forward(self, wave: torch.Tensor) -> torch.Tensor:
        if wave.ndim == 1:
            wave = wave.unsqueeze(0)
        mel = self.mel(wave)
        mel = self.db(mel)
        mel = mel.unsqueeze(1)
        mel = F.interpolate(mel, size=CFG.image_size, mode='bilinear', align_corners=False)
        flat = mel.flatten(start_dim=1)
        min_val = flat.min(dim=1).values[:, None, None, None]
        max_val = flat.max(dim=1).values[:, None, None, None]
        mel = (mel - min_val) / (max_val - min_val + 1e-7)
        return mel


def build_model() -> nn.Module:
    return timm.create_model(
        CFG.model_name,
        pretrained=False,
        in_chans=1,
        num_classes=len(CLASSES),
        drop_path_rate=0.0,
    )


def load_models(fold_paths: dict[int, Path]) -> list[nn.Module]:
    models: list[nn.Module] = []
    for fold_id in sorted(fold_paths):
        payload = torch.load(fold_paths[fold_id], map_location='cpu')
        model = build_model()
        model.load_state_dict(payload['model_state_dict'])
        model.to(device)
        model.eval()
        models.append(model)
    return models


fold_paths = resolve_fold_checkpoint_paths(MODEL_DATASET_HINT)
models = load_models(fold_paths)
mel_transform = LogMelSpectrogramTransform().to(device).eval()
print({
    'fold_paths': {fold: str(path) for fold, path in fold_paths.items()},
    'models_loaded': len(models),
})


In [ ]:
def load_soundscape(path: Path) -> np.ndarray:
    audio, _ = librosa.load(path, sr=CFG.sample_rate, mono=True)
    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
    return audio


def extract_chunks(audio: np.ndarray, end_seconds: list[int]) -> np.ndarray:
    chunks = np.zeros((len(end_seconds), CFG.chunk_samples), dtype=np.float32)
    for idx, end_second in enumerate(end_seconds):
        end_sample = int(end_second * CFG.sample_rate)
        start_sample = max(0, end_sample - CFG.chunk_samples)
        segment = audio[start_sample:end_sample]
        if segment.shape[0] >= CFG.chunk_samples:
            chunks[idx] = segment[-CFG.chunk_samples:]
        else:
            chunks[idx, :segment.shape[0]] = segment
    return chunks


def predict_chunks(chunks: np.ndarray, models: list[nn.Module]) -> np.ndarray:
    preds = []
    for start in range(0, len(chunks), CFG.batch_chunks):
        batch_np = chunks[start: start + CFG.batch_chunks]
        batch = torch.from_numpy(batch_np).to(device=device, dtype=torch.float32)
        with torch.no_grad():
            images = mel_transform(batch)
            fold_probs = []
            for model in models:
                with autocast_context():
                    logits = model(images)
                probs = torch.sigmoid(logits).float()
                fold_probs.append(probs)
            mean_probs = torch.stack(fold_probs, dim=0).mean(dim=0)
        preds.append(mean_probs.detach().cpu().numpy().astype(np.float32)[:, submission_class_indices])
        del batch, images, fold_probs, mean_probs
    return np.concatenate(preds, axis=0) if preds else np.zeros((0, len(submission_species)), dtype=np.float32)


def build_submission(row_ids: list[str], preds: np.ndarray, expected_ids: list[str], species: list[str]) -> tuple[pd.DataFrame, int]:
    if len(row_ids) == 0:
        pred_df = pd.DataFrame(columns=['row_id', *species])
    else:
        pred_df = pd.DataFrame(preds, columns=species)
        pred_df.insert(0, 'row_id', row_ids)
        pred_df = pred_df.drop_duplicates(subset='row_id', keep='first')

    missing_ids = [row_id for row_id in expected_ids if row_id not in set(pred_df['row_id'].tolist())]
    if missing_ids:
        filler = pd.DataFrame({'row_id': missing_ids})
        for label in species:
            filler[label] = 0.0
        pred_df = pd.concat([pred_df, filler], axis=0, ignore_index=True)

    pred_df = pred_df.set_index('row_id').loc[expected_ids].reset_index()
    return pred_df, len(missing_ids)


In [ ]:
all_row_ids: list[str] = []
all_preds: list[np.ndarray] = []

for stem, path in tqdm(sorted(test_file_map.items()), desc='soundscapes'):
    end_seconds = active_expected_seconds.get(stem, [])
    if not end_seconds:
        continue
    audio = load_soundscape(path)
    chunks = extract_chunks(audio, end_seconds)
    pred = predict_chunks(chunks, models)
    row_ids = [f'{stem}_{second}' for second in end_seconds]
    all_row_ids.extend(row_ids)
    all_preds.append(pred)

pred_matrix = np.concatenate(all_preds, axis=0) if all_preds else np.zeros((0, len(submission_species)), dtype=np.float32)
submission, missing_count = build_submission(
    row_ids=all_row_ids,
    preds=pred_matrix,
    expected_ids=active_sample_sub['row_id'].tolist(),
    species=submission_species,
)

submission.to_csv(SUBMISSION_PATH, index=False)
print({
    'submission_path': str(SUBMISSION_PATH),
    'rows': int(len(submission)),
    'missing_row_ids_filled_with_zeros': int(missing_count),
    'used_test_fallback': used_test_fallback,
})
submission.head()
